In [ ]:
# ===== Cell 1: Install Ollama, start server, pull model =====
import subprocess, time

MODEL_NAME = "deepseek-coder-v2:16b"

subprocess.run("apt-get install -y zstd > /dev/null 2>&1", shell=True)
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True)
subprocess.run("pip install -q ollama pyyaml", shell=True)

# Start the Ollama server in the background
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

# Download the model (~9 GB, takes a few minutes)
subprocess.run(["ollama", "pull", MODEL_NAME], check=True)


# ===== Cell 2: Generator code =====
import re
import yaml
from ollama import chat

SERVICE_NAME = "GitHub Actions"


def extract_yaml_only(text: str) -> str:
    text = text.strip()
    match = re.search(r"```(?:ya?ml)?\s*\n(.*?)```", text, flags=re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(1).strip()
    text = re.sub(r"^```(yml|yaml)?", "", text, flags=re.IGNORECASE)
    text = re.sub(r"```.*", "", text, flags=re.DOTALL)
    return text.strip()


def is_valid_yaml(text: str) -> bool:
    try:
        return isinstance(yaml.safe_load(text), dict)
    except yaml.YAMLError:
        return False


def generate_yaml(description: str, model_name: str = MODEL_NAME,
                  context_len: int = 8192, max_retries: int = 2) -> str:
    messages = [
        {"role": "system",
         "content": "You are a highly skilled software developer and DevOps engineer"},
        {"role": "user",
         "content": (
             f"Given the description below, generate a minimal and single valid {SERVICE_NAME} "
             "YAML configuration. Do not provide any reasoning. Only output the single YAML "
             "configuration, with no comments, markdown, or extra formatting. The YAML must "
             "match the described task exactly. Do not add any unneeded directives unless told."
             f"\n\n{description}"
         )},
    ]

    yaml_output = ""
    for attempt in range(max_retries + 1):
        response = chat(
            model=model_name,
            messages=messages,
            options={"temperature": 0, "num_ctx": context_len, "num_predict": 4096},
        )
        yaml_output = extract_yaml_only(response.message.content or "")
        if is_valid_yaml(yaml_output):
            return yaml_output
        print(f"Attempt {attempt + 1}: invalid YAML, retrying...")
        messages.append({"role": "assistant", "content": response.message.content or ""})
        messages.append({"role": "user",
                         "content": "That was not valid YAML. Output only the corrected YAML."})
    return yaml_output


# ===== Cell 3: Run it =====
description = """Once you've completed the prerequisites, you can proceed with creating the workflow. The following example workflow demonstrates how to build and deploy a .NET project to Azure App Service when there is a push to the main branch. Ensure that you set AZURE_WEBAPP_NAME in the workflow env key to the name of the web app you created. If the path to your project is not the repository root, change AZURE_WEBAPP_PACKAGE_PATH . If you use a version of .NET other than 5 , change DOTNET_VERSION . If you configured a deployment environment, change the value of environment to be the name of your environment. If you did not configure an environment or if your workflow is in a private repository and you do not use GitHub Enterprise Cloud, delete the environment key."""

# If you hit an out-of-memory error, use context_len=4096
result = generate_yaml(description, model_name=MODEL_NAME, context_len=8192)
print(result)

with open("workflow_deepseek.yml", "w") as f:
    f.write(result)

name: CI

on:
  push:
    branches:
      - main

jobs:
  build:
    runs-on: ubuntu-latest

    steps:
    - uses: actions/checkout@v2
    - name: Set up .NET Core SDK
      uses: actions/setup-dotnet@v1
      with:
        dotnet-version: '5.0.x'
    - name: Install dependencies
      run: dotnet restore
    - name: Build
      run: dotnet build --configuration Release
    - name: Publish
      run: dotnet publish -c Release -o ./publish
    - name: Upload Artifact
      uses: actions/upload-artifact@v2
      with:
        name: package
        path: ./publish

  deploy:
    runs-on: ubuntu-latest
    needs: build

    steps:
    - uses: actions/checkout@v2
    - name: Download Artifact
      uses: actions/download-artifact@v2
      with:
        name: package
    - name: Set up Azure CLI
      uses: azure/setup-azcli@v1
    - name: Login to Azure
      uses: azure/login@v1
      with:
        creds: ${{ secrets.AZURE_CREDENTIALS }}
    - name: Deploy to Azure App Service
      uses:

In [ ]:
# ===== Cell 1: Install Ollama, start server, pull model =====
import subprocess, time

MODEL_NAME = "deepseek-coder-v2:16b"

subprocess.run("apt-get install -y zstd > /dev/null 2>&1", shell=True)
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True)
subprocess.run("pip install -q ollama pyyaml", shell=True)

# Start the Ollama server in the background
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

# Download the model (~9 GB, takes a few minutes)
subprocess.run(["ollama", "pull", MODEL_NAME], check=True)


# ===== Cell 2: Generator code =====
import re
import yaml
from ollama import chat

SERVICE_NAME = "GitHub Actions"


def extract_yaml_only(text: str) -> str:
    text = text.strip()
    match = re.search(r"```(?:ya?ml)?\s*\n(.*?)```", text, flags=re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(1).strip()
    text = re.sub(r"^```(yml|yaml)?", "", text, flags=re.IGNORECASE)
    text = re.sub(r"```.*", "", text, flags=re.DOTALL)
    return text.strip()


def is_valid_yaml(text: str) -> bool:
    try:
        return isinstance(yaml.safe_load(text), dict)
    except yaml.YAMLError:
        return False


def generate_yaml(description: str, model_name: str = MODEL_NAME,
                  context_len: int = 8192, max_retries: int = 2) -> str:
    messages = [
        {"role": "system",
         "content": "You are a highly skilled software developer and DevOps engineer"},
        {"role": "user",
         "content": (
             f"Given the description below, generate a minimal and single valid {SERVICE_NAME} "
             "YAML configuration. Do not provide any reasoning. Only output the single YAML "
             "configuration, with no comments, markdown, or extra formatting. The YAML must "
             "match the described task exactly. Do not add any unneeded directives unless told."
             f"\n\n{description}"
         )},
    ]

    yaml_output = ""
    for attempt in range(max_retries + 1):
        response = chat(
            model=model_name,
            messages=messages,
            options={"temperature": 0, "num_ctx": context_len, "num_predict": 4096},
        )
        yaml_output = extract_yaml_only(response.message.content or "")
        if is_valid_yaml(yaml_output):
            return yaml_output
        print(f"Attempt {attempt + 1}: invalid YAML, retrying...")
        messages.append({"role": "assistant", "content": response.message.content or ""})
        messages.append({"role": "user",
                         "content": "That was not valid YAML. Output only the corrected YAML."})
    return yaml_output


# ===== Cell 3: Run it (change the description here for each new one) =====
description = """This section briefly describes some of the advanced features of GitHub Actions that help you create more complex workflows. If your workflows use sensitive data, such as passwords or certificates, you can save these in GitHub as secrets and then use them in your workflows as environment variables. This means that you will be able to create and share workflows without having to embed sensitive values directly in the workflow's YAML source. This example job demonstrates how to reference an existing secret as an environment variable, and send it as a parameter to an example command."""

# If you hit an out-of-memory error, use context_len=4096
result = generate_yaml(description, model_name=MODEL_NAME, context_len=8192)
print(result)

with open("workflow_deepseek.yml", "w") as f:
    f.write(result)

name: Example Workflow

on: [push]

jobs:
  example-job:
    runs-on: ubuntu-latest
    steps:
      - name: Checkout repository
        uses: actions/checkout@v2
      - name: Set environment variable from secret
        env:
          SECRET_VAR: ${{ secrets.YOUR_SECRET_NAME }}
        run: echo $SECRET_VAR
      - name: Example command with secret parameter
        run: echo "The secret value is $SECRET_VAR"


In [ ]:
description = """This section briefly describes some of the advanced features of GitHub Actions that help you create more complex workflows. By default, the jobs in your workflow all run in parallel at the same time. If you have a job that must only run after another job has completed, you can use the needs keyword to create this dependency. If one of the jobs fails, all dependent jobs are skipped; however, if you need the jobs to continue, you can define this using the if conditional statement. In this example, the setup , build , and test jobs run in series, with build and test being dependent on the successful completion of the job that precedes them:"""

result = generate_yaml(description, model_name=MODEL_NAME, context_len=8192)
print(result)

with open("workflow_deepseek.yml", "w") as f:
    f.write(result)

name: CI

on:
  push:
    branches: [ "main" ]
  pull_request:
    branches: [ "main" ]

jobs:
  setup:
    runs-on: ubuntu-latest
    steps:
      - name: Checkout repository
        uses: actions/checkout@v2

  build:
    runs-on: ubuntu-latest
    needs: setup
    steps:
      - name: Checkout repository
        uses: actions/checkout@v2
      - name: Build
        run: |
          echo "Building the project..."

  test:
    runs-on: ubuntu-latest
    needs: build
    steps:
      - name: Checkout repository
        uses: actions/checkout@v2
      - name: Run tests
        run: |
          echo "Running tests..."


In [ ]:
description = """This section briefly describes some of the advanced features of GitHub Actions that help you create more complex workflows. A matrix strategy lets you use variables in a single job definition to automatically create multiple job runs that are based on the combinations of the variables. For example, you can use a matrix strategy to test your code in multiple versions of a language or on multiple operating systems. The matrix is created using the strategy keyword, which receives the build options as an array. For example, this matrix will run the job multiple times, using different versions of Node.js:"""

result = generate_yaml(description, model_name=MODEL_NAME, context_len=8192)
print(result)

with open("workflow_deepseek.yml", "w") as f:
    f.write(result)

name: CI

on:
  push:
    branches: [ "main" ]
  pull_request:
    branches: [ "main" ]

jobs:
  build:
    runs-on: ubuntu-latest
    strategy:
      matrix:
        node-version: [12.x, 14.x, 16.x]
    steps:
      - uses: actions/checkout@v2
      - name: Use Node.js ${{ matrix.node-version }}
        uses: actions/setup-node@v1
        with:
          node-version: ${{ matrix.node-version }}
      - run: npm install
      - run: npm test


In [ ]:
description = """This section briefly describes some of the advanced features of GitHub Actions that help you create more complex workflows. If your jobs regularly reuse dependencies, you can consider caching these files to help improve performance. Once the cache is created, it is available to all workflows in the same repository. This example demonstrates how to cache the ~/.npm directory:"""

result = generate_yaml(description, model_name=MODEL_NAME, context_len=8192)
print(result)

with open("workflow_deepseek.yml", "w") as f:
    f.write(result)

jobs:
  build:
    runs-on: ubuntu-latest
    steps:
    - uses: actions/checkout@v2
    - name: Set up Node.js
      uses: actions/setup-node@v1
      with:
        node-version: '14'
    - name: Cache Node.js modules
      uses: actions/cache@v2
      with:
        path: ~/.npm
        key: ${{ runner.os }}-node-${{ hashFiles('**/package-lock.json') }}
        restore-keys: |
          ${{ runner.os }}-node-
    - name: Install dependencies
      run: npm install
    - name: Run tests
      run: npm test


In [ ]:
description = """This section briefly describes some of the advanced features of GitHub Actions that help you create more complex workflows. If your job requires a database or cache service, you can use the services keyword to create an ephemeral container to host the service; the resulting container is then available to all steps in that job and is removed when the job has completed. This example demonstrates how a job can use services to create a postgres container, and then use node to connect to the service."""

result = generate_yaml(description, model_name=MODEL_NAME, context_len=8192)
print(result)

with open("workflow_deepseek.yml", "w") as f:
    f.write(result)

name: CI

on:
  push:
    branches: [ "main" ]
  pull_request:
    branches: [ "main" ]

jobs:
  build:
    runs-on: ubuntu-latest

    services:
      postgres:
        image: postgres
        env:
          POSTGRES_USER: postgres
          POSTGRES_PASSWORD: postgres
        ports:
          - 5432:5432
        options: --health-cmd="pg_isready" --health-interval=10s --health-timeout=5s --health-retries=3

    steps:
      - name: Checkout code
        uses: actions/checkout@v2

      - name: Install Node.js
        uses: actions/setup-node@v1
        with:
          node-version: '14'

      - name: Install dependencies
        run: npm install

      - name: Connect to PostgreSQL
        run: npm run test-db

      - name: Run tests
        run: npm test


In [ ]:
description = """This section briefly describes some of the advanced features of GitHub Actions that help you create more complex workflows. If you want to be sure that a particular type of runner will process your job, you can use labels to control where jobs are executed. You can assign labels to a self-hosted runner in addition to their default label of self-hosted . Then, you can refer to these labels in your YAML workflow, ensuring that the job is routed in a predictable way. GitHub-hosted runners have predefined labels assigned. This example shows how a workflow can use labels to specify the required runner:"""

result = generate_yaml(description, model_name=MODEL_NAME, context_len=8192)
print(result)

with open("workflow_deepseek.yml", "w") as f:
    f.write(result)

name: CI

on:
  push:
    branches: [ "main" ]
  pull_request:
    branches: [ "main" ]

jobs:
  build:
    runs-on: [self-hosted, linux]
    steps:
      - name: Checkout repository
        uses: actions/checkout@v2
      - name: Set up Python
        uses: actions/setup-python@v2
        with:
          python-version: '3.x'
      - name: Install dependencies
        run: |
          python -m pip install --upgrade pip
          pip install -r requirements.txt
      - name: Run tests
        run: |
          pytest


In [ ]:
description = """You can use GitHub Actions to generate artifact attestations that establish build provenance for artifacts such as binaries and container images. To generate an artifact attestation, you must: When you run your updated workflows, they will build your artifacts and generate an artifact attestation that establishes build provenance. You can view attestations in your repository's Actions tab. For more information, see the attest-build-provenance repository. In the workflow that builds the binary you would like to attest, add the following permissions."""

result = generate_yaml(description, model_name=MODEL_NAME, context_len=8192)
print(result)

with open("workflow_deepseek.yml", "w") as f:
    f.write(result)

name: Generate Artifact Attestation

on:
  push:
    branches:
      - main

permissions:
  contents: read
  security-events: write

jobs:
  build-and-attest:
    runs-on: ubuntu-latest

    steps:
      - name: Checkout repository
        uses: actions/checkout@v3

      - name: Set up Go
        uses: actions/setup-go@v3
        with:
          go-version: 1.18

      - name: Install dependencies
        run: go get -d ./...

      - name: Build binary
        run: go build -o my-binary

      - name: Generate artifact attestation
        run: |
          echo "Generating attestation for my-binary"
          # Add commands to generate the attestation here

      - name: Upload artifact attestation
        uses: actions/upload-artifact@v3
        with:
          name: my-binary-attestation
          path: ./my-binary-attestation.txt


In [ ]:
description = """Using artifact attestations to establish provenance for builds - Generating artifact attestations for your builds - Generating build provenance for binaries"""

result = generate_yaml(description, model_name=MODEL_NAME, context_len=8192)
print(result)

with open("workflow_deepseek.yml", "w") as f:
    f.write(result)

name: Build Provenance

on:
  push:
    branches:
      - main

jobs:
  build:
    runs-on: ubuntu-latest

    steps:
      - name: Checkout code
        uses: actions/checkout@v2

      - name: Set up Go
        uses: actions/setup-go@v2
        with:
          go-version: 1.16

      - name: Install dependencies
        run: go get -d ./...

      - name: Build
        run: go build

      - name: Generate artifact attestation
        uses: sigstore/cosign-action@v1
        with:
          image: your-docker-image-name
          key: ${{ secrets.COSIGN_KEY }}
          output: attestation.json

      - name: Upload artifact attestation
        uses: actions/upload-artifact@v2
        with:
          name: attestation
          path: attestation.json


In [ ]:
# ===== Cell 1: Install Ollama, start server, pull model =====
import subprocess, time

MODEL_NAME = "deepseek-coder-v2:16b"

subprocess.run("apt-get install -y zstd > /dev/null 2>&1", shell=True)
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True)
subprocess.run("pip install -q ollama pyyaml", shell=True)

# Start the Ollama server in the background
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

# Download the model (~9 GB, takes a few minutes)
subprocess.run(["ollama", "pull", MODEL_NAME], check=True)


# ===== Cell 2: Generator code =====
import re
import yaml
from ollama import chat

SERVICE_NAME = "GitHub Actions"


def extract_yaml_only(text: str) -> str:
    text = text.strip()
    match = re.search(r"```(?:ya?ml)?\s*\n(.*?)```", text, flags=re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(1).strip()
    text = re.sub(r"^```(yml|yaml)?", "", text, flags=re.IGNORECASE)
    text = re.sub(r"```.*", "", text, flags=re.DOTALL)
    return text.strip()


def is_valid_yaml(text: str) -> bool:
    try:
        return isinstance(yaml.safe_load(text), dict)
    except yaml.YAMLError:
        return False


def generate_yaml(description: str, model_name: str = MODEL_NAME,
                  context_len: int = 8192, max_retries: int = 2) -> str:
    messages = [
        {"role": "system",
         "content": "You are a highly skilled software developer and DevOps engineer"},
        {"role": "user",
         "content": (
             f"Given the description below, generate a minimal and single valid {SERVICE_NAME} "
             "YAML configuration. Do not provide any reasoning. Only output the single YAML "
             "configuration, with no comments, markdown, or extra formatting. The YAML must "
             "match the described task exactly. Do not add any unneeded directives unless told."
             f"\n\n{description}"
         )},
    ]

    yaml_output = ""
    for attempt in range(max_retries + 1):
        response = chat(
            model=model_name,
            messages=messages,
            options={"temperature": 0, "num_ctx": context_len, "num_predict": 4096},
        )
        yaml_output = extract_yaml_only(response.message.content or "")
        if is_valid_yaml(yaml_output):
            return yaml_output
        print(f"Attempt {attempt + 1}: invalid YAML, retrying...")
        messages.append({"role": "assistant", "content": response.message.content or ""})
        messages.append({"role": "user",
                         "content": "That was not valid YAML. Output only the corrected YAML."})
    return yaml_output


# ===== Cell 3: Run it (change the description here for each new one) =====
description = """You can use GitHub Actions to generate artifact attestations that establish build provenance for artifacts such as binaries and container images. To generate an artifact attestation, you must: When you run your updated workflows, they will build your artifacts and generate an artifact attestation that establishes build provenance. You can view attestations in your repository's Actions tab. For more information, see the attest-build-provenance repository. In the workflow that builds the container image you would like to attest, add the following permissions."""

# If you hit an out-of-memory error, use context_len=4096
result = generate_yaml(description, model_name=MODEL_NAME, context_len=8192)
print(result)

with open("workflow_deepseek.yml", "w") as f:
    f.write(result)

name: Generate Artifact Attestation

on:
  push:
    branches:
      - main

permissions:
  contents: read
  security-events: write

jobs:
  build-and-attest:
    runs-on: ubuntu-latest

    steps:
      - name: Checkout repository
        uses: actions/checkout@v2

      - name: Set up Docker Buildx
        uses: docker/setup-buildx-action@v1

      - name: Build container image
        run: docker build -t my-image .

      - name: Generate artifact attestation
        run: |
          echo "Generating attestation for my-image"
          # Add commands to generate the attestation here

      - name: Upload artifact attestation
        uses: actions/upload-artifact@v2
        with:
          name: my-image-attestation
          path: ./attestation.json
